In [ ]:
!pip install pypdf langchain langchain-community langchain-core langchain-google-genai langchain-huggingface faiss-cpu sentence-transformers

In [ ]:
!pip install anthropic pypdf langchain-text-splitters langchain-huggingface langchain-community faiss-cpu sentence-transformers

In [ ]:
!pip install groq pypdf langchain-text-splitters langchain-huggingface langchain-community faiss-cpu sentence-transformers

In [ ]:
import os
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from groq import Groq

# =========================
# GOOGLE DRIVE MOUNT
# =========================
try:
    from google.colab import drive
    print(" Mounting Google Drive...")
    drive.mount('/content/drive')
    print("Google Drive Mounted Successfully!")
except ImportError:
    print("Not running in Google Colab. Local directory will be used.")

# =========================
# API KEY SETUP
# =========================
# FREE API Key: https://console.groq.com/
os.environ["GROQ_API_KEY"] = "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"

DB_FAISS_PATH = "/content/drive/MyDrive/faiss_index_db"

# =========================
# LOAD PDF
# =========================
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text
    return text

# =========================
# CHUNKING
# =========================
def split_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )
    return splitter.split_text(text)

# =========================
# VECTOR DB MANAGEMENT (Save & Load)
# =========================
def get_vector_store(pdf_path):
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    if os.path.exists(DB_FAISS_PATH):
        print(" Loading existing FAISS index from Google Drive... ")
        db = FAISS.load_local(DB_FAISS_PATH, embeddings, allow_dangerous_deserialization=True)
    else:
        print(" No existing index found. Processing PDF for the first time...")
        print(" Loading PDF...")
        text = load_pdf(pdf_path)

        print(" Chunking text...")
        chunks = split_text(text)
        print("Total chunks:", len(chunks))

        print(" Creating embeddings + FAISS DB...")
        db = FAISS.from_texts(chunks, embeddings)

        print(" Saving FAISS index to Google Drive for next time...")
        db.save_local(DB_FAISS_PATH)
        print("Index saved!")

    return db

# =========================
# GROQ API CALL
# =========================
def call_groq(system_prompt, user_message):
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    chat_completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",  # Free model, fast & smart
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.7,
        max_tokens=1024,
    )
    return chat_completion.choices[0].message.content

# =========================
# PROMPTS
# =========================
RAG_SYSTEM_PROMPT = """You are a helpful AI assistant.
Use ONLY the given context to answer the question.
If the answer is not in the context, say "I don't know based on the document only." """

GREETING_SYSTEM_PROMPT = """You are a friendly, welcoming, and smart AI assistant.
Acknowledge the user's greeting warmly and naturally.
Tell them you are ready to help analyze their uploaded document."""

# =========================
# MAIN PIPELINE
# =========================
def run_rag(pdf_path):
    db = get_vector_store(pdf_path)
    retriever = db.as_retriever(search_kwargs={"k": 3})

    print("\n🤖 RAG Chatbot Ready! (Powered by Groq - Free!)\n")

    greetings = [
        "hi", "hello", "hey", "greetings", "how are you",
        "what's up", "who are you", "what is your name",
        "good morning", "good afternoon", "lets start", "let's start"
    ]

    while True:
        query = input("Ask: ").strip()

        if not query:
            continue

        if query.lower() in ["exit", "quit"]:
            print("👋 Goodbye!")
            break

        print(" Thinking...\n")

        is_greeting = any(greet in query.lower() for greet in greetings)

        if is_greeting:
            response_text = call_groq(GREETING_SYSTEM_PROMPT, query)
        else:
            docs = retriever.invoke(query)
            context = "\n\n".join([d.page_content for d in docs])
            user_message = f"Context:\n{context}\n\nQuestion:\n{query}"
            response_text = call_groq(RAG_SYSTEM_PROMPT, user_message)

        print("\n🤖 Answer:\n")
        print(response_text)
        print("\n" + "-" * 50)

# =========================
# RUN
# =========================
pdf_path = "/content/rimsha.pdf"
run_rag(pdf_path)